# SemRestore-QA — run everything on Colab

**Runtime → Change runtime type → T4 GPU.** Run the cells top to bottom.

Everything is saved to Google Drive, so if Colab disconnects just run the setup cells again and continue:
every step skips work that is already done.

| Step | What it does | Time on a T4 (300 images) |
|---|---|---|
| 01 | pick COCO images + object masks, split dev/test | ~10 min |
| 02 | edits: addition, removal, substitution, recolor | ~45–60 min |
| 03 | PSNR-matched blur/noise controls | ~5 min |
| 04 | x4 super-resolution, 3 families | ~60 min (diffusion is the slow one) |
| 05 | score everything with every metric | ~20–30 min |
| 06 | calibrate + tune on dev, **freeze** | 1 min |
| 07 | final evaluation on held-out test | 2–5 min |
| 08 | build the human study page | 1 min |
| 09 | analyse human ratings | 1 min |
| 10 | figures | 1 min |

Times are rough estimates: each step prints how long its first item took, so check that early.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/semrestore-qa'   # <- change if you like

Get the code. Either upload `semrestore-qa.zip` to the top level of **My Drive** (it unzips into the folder above), or clone your team's GitHub repo.

In [ ]:
import os, zipfile
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f'{WORK}/config.yaml') and os.path.exists(f'{WORK}.zip'):
    zipfile.ZipFile(f'{WORK}.zip').extractall(os.path.dirname(WORK))
# or:  !git clone https://github.com/<your-team>/semrestore-qa.git {WORK}
%cd {WORK}
!ls

In [ ]:
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

Optional sanity check (≈3 min on GPU): unit tests, then the whole pipeline on synthetic images with random networks.

In [ ]:
!python -m pytest -q tests
# !python scripts/00_smoke_test.py --out /content/smoke_run

## 01–04: build the benchmark
Tip: first try `--limit 5` on step 02 and look at `results/figures/edit_samples.png` before running all images.

In [ ]:
!python scripts/01_build_dataset.py

In [ ]:
!python scripts/02_make_edits.py --limit 5
from IPython.display import Image; Image('results/figures/edit_samples.png')

In [ ]:
!python scripts/02_make_edits.py

In [ ]:
!python scripts/03_make_controls.py

In [ ]:
# one family at a time is fine (and safer on free Colab)
!python scripts/04_restore.py --families regression gan
!python scripts/04_restore.py --families diffusion

## 05–07: score, freeze on dev, evaluate on held-out test

In [ ]:
!python scripts/05_score.py

In [ ]:
!python scripts/06_freeze.py

In [ ]:
!python scripts/07_evaluate.py
from IPython.display import Markdown; Markdown(open('results/tables/summary.md').read())

## 08–09: human study
08 writes `results/human/rating.html`. Send that one file to 5–8 classmates. They open it in a browser, rate, and send back the CSV it downloads. Put the CSVs in `results/human/ratings/`, then run 09.

In [ ]:
!python scripts/08_human_pairs.py

In [ ]:
!python scripts/09_analyze_human.py

## 10: figures

In [ ]:
!python scripts/10_figures.py
from IPython.display import Image, display
for f in ['fig1_examples', 'fig2_winrate', 'fig3_sharpness_bias', 'fig4_cross_restoration', 'fig5_restorers']:
    p = f'results/figures/{f}.png'
    if os.path.exists(p): display(Image(p))